In [ ]:
# Larger CNN for the MNIST Dataset
import numpy
from keras.datasets import mnist
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import Dropout
from keras.layers import Flatten
from keras.layers.convolutional import Convolution2D
from keras.layers.convolutional import MaxPooling2D
from keras.utils import np_utils
import os,cv2
import numpy as np
import matplotlib.pyplot as plt

from sklearn.utils import shuffle
#from sklearn.cross_validation import train_test_split
from sklearn.model_selection import train_test_split
from keras import backend as K
#K.set_image_dim_ordering('th')

from keras.utils import np_utils
from keras.models import Sequential
from keras.layers.core import Dense, Dropout, Activation, Flatten
from keras.layers.convolutional import Convolution2D, MaxPooling2D
from keras.optimizers import SGD,RMSprop



In [ ]:

import cv2
img_rows=224
img_cols=224
num_channel=3
num_epoch=50

# Define the number of classes
num_classes = 5

labels_name={'No_DR':0,'Mild':1,'Moderate':2,'Severe':3,'Proliferate_DR':4}



In [ ]:

path=os.getcwd()
data=os.listdir("{}//dataset1".format(path))


In [ ]:
img_data_list=[]
labels_list = []
for dataset in data:
    img_list=os.listdir("{}//dataset1//{}".format(path,dataset))
    print ('Loading the images of dataset-'+'{}\n'.format(dataset))
    label = labels_name[dataset]
    for img in img_list:
        print(img)
        input_img=cv2.imread("{}//dataset1//{}//{}".format(path,dataset,img))
        #input_img=cv2.cvtColor(input_img, cv2.COLOR_BGR2GRAY) 

        input_img_resize=cv2.resize(input_img,(224,224))
        print(input_img_resize.shape)
        #input_img= np.expand_dims(input_img_resize, axis=4)
        #print(input_img)
        img_data_list.append(input_img_resize)
        labels_list.append(label)

img_data = np.array(img_data_list)
print(img_data.shape)
img_data = img_data.astype('float32')
img_data /= 255
#print ("img_data_shape:",img_data.shape)
x=img_data
labels = np.array(labels_list)
# print the count of number of samples for different classes
print(np.unique(labels,return_counts=True))
# convert class labels to on-hot encoding
Y = np_utils.to_categorical(labels, num_classes)
y=Y
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=2)




In [ ]:
print(X_train.shape)
X_train = X_train.reshape(X_train.shape[0],224, 224,3).astype( 'float32' )
X_test = X_test.reshape(X_test.shape[0],224, 224,3).astype( 'float32' )

In [ ]:
# fix random seed for reproducibility
from sklearn.model_selection import train_test_split

seed = 7
numpy.random.seed(seed)
# load data
#x=img_data
#y=Y
#x,y = shuffle(img_data,Y, random_state=2)
# Split the dataset
#X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=2)
#(X_train, y_train), (X_test, y_test) = 
# reshape to be [samples][pixels][width][height]


In [ ]:
# Defining the model
# Defining the model
import keras
input_shape=(224,224,3)
print("input_shape",input_shape)					
#Instantiate an empty model
model = Sequential()

# C1 Convolutional Layer
model.add(keras.layers.Conv2D(6, kernel_size=(5, 5), strides=(1, 1), activation='tanh', input_shape=(224,224,3), padding='same'))

# S2 Pooling Layer
model.add(keras.layers.AveragePooling2D(pool_size=(2, 2), strides=(1, 1), padding='valid'))

# C3 Convolutional Layer
model.add(keras.layers.Conv2D(16, kernel_size=(5, 5), strides=(1, 1), activation='tanh', padding='valid'))

# S4 Pooling Layer
model.add(keras.layers.AveragePooling2D(pool_size=(2, 2), strides=(2, 2), padding='valid'))

# C5 Fully Connected Convolutional Layer
model.add(keras.layers.Conv2D(120, kernel_size=(5, 5), strides=(1, 1), activation='tanh', padding='valid'))
#Flatten the CNN output so that we can connect it with fully connected layers
model.add(keras.layers.Flatten())

# FC6 Fully Connected Layer
model.add(keras.layers.Dense(84, activation='tanh'))

#Output Layer with softmax activation
model.add(keras.layers.Dense(num_classes, activation='softmax'))

# Compile the model
model.compile(loss=keras.losses.categorical_crossentropy, optimizer='SGD', metrics=['accuracy'])
# Viewing model_configuration

model.summary()



In [ ]:

#%%
# Training
hist = model.fit(X_train, y_train, batch_size=16,epochs = 25,verbose=1, validation_data=(X_test, y_test))

#hist = model.fit(X_train, y_train, batch_size=32, nb_epoch=20,verbose=1, validation_split=0.2)

# Training with callbacks
from keras import callbacks

#filename='model_train_new.csv'
#csv_log=callbacks.CSVLogger(filename, separator=',', append=False)

#early_stopping=callbacks.EarlyStopping(monitor='val_loss', min_delta=0, patience=0, verbose=0, mode='min')

#filepath="Best-weights-my_model-{epoch:03d}-{loss:.4f}-{acc:.4f}.hdf5"

#checkpoint = callbacks.ModelCheckpoint(filepath, monitor='val_loss', verbose=1, mode='min')

#callbacks_list = [csv_log,checkpoint]



# visualizing losses and accuracy
train_loss=hist.history['loss']
val_loss=hist.history['val_loss']

In [ ]:
#%%
# Printing the confusion matrix
from sklearn.metrics import classification_report,confusion_matrix
import itertools

Y_pred = model.predict(X_test)
print(Y_pred)
y_pred = np.argmax(Y_pred, axis=1)
print(y_pred)
#y_pred = model.predict_classes(X_test)
#print(y_pred)
target_names =['class 0(No_DR)','class 1(Mild)','class 2(Moderate)','class 3(Severe)','class 4(Proliferate_DR)']
					
print(classification_report(np.argmax(y_test,axis=1), y_pred,target_names=target_names))

print(confusion_matrix(np.argmax(y_test,axis=1), y_pred))


# Plotting the confusion matrix
def plot_confusion_matrix(cm, classes,
                          normalize=False,
                          title='Confusion matrix',
                          cmap=plt.cm.Blues):
    """
    This function prints and plots the confusion matrix.
    Normalization can be applied by setting `normalize=True`.
    """
    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)

    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        print("Normalized confusion matrix")
    else:
        print('Confusion matrix, without normalization')

    print(cm)

    thresh = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, cm[i, j],
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")

    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predicted label')


In [ ]:
# Testing a new image
test_image1 = cv2.imread("assets/Sample.jpeg")
#test_image=cv2.cvtColor(test_image, cv2.COLOR_BGR2GRAY)
test_image2=cv2.resize(test_image1,(224,224))
test_image = test_image2.reshape(1,224, 224,3).astype( 'float32' )


test_image = np.array(test_image)
test_image = test_image.astype('float32')
test_image /= 255
print (test_image.shape)
#test_image = test_image.reshape(1,224, 224,3).astype( 'float32' )
res=model.predict(test_image)
print(res)
print( np.argmax(res,axis=1))
# print(model.predict_classes(test_image))


In [ ]:
model.save('model.h5')
